# Preprocessing - Hotel Booking Demand

Pipeline complet, du CSV brut jusqu'aux jeux prets pour la modelisation, plus un fichier de
reference metier (prix des chambres en euros) pour l'analyse de couts ulterieure.

Principe : on nettoie d'abord (operations sures), on **splitte** ensuite, puis on applique les
transformations qui apprennent des donnees (encodages, scaling) sur le train uniquement, pour
eviter toute fuite de donnees.

## 0. Imports

In [73]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

## 1. Import des donnees

In [74]:
DATA_PATH = "data/hotel_bookings.csv"   # a adapter
df = pd.read_csv(DATA_PATH)
print("Dimensions brutes :", df.shape)   # (119390, 36)

Dimensions brutes : (119390, 32)


## 2. Suppression de colonnes

Quatre familles a retirer avant tout, car elles ne doivent pas entrer dans le modele :

- **Fausses donnees perso** (`name`, `email`, `phone-number`, `credit_card`) : donnees
  synthetiques, aucune valeur predictive.
- **Fuite de cible** (`reservation_status`, `reservation_status_date`) : correspondance parfaite
  avec `is_canceled`, connue apres l'evenement.
- **`deposit_type`** : 99,4 % d'annulation sur "Non Refund" et chi2 le plus eleve (27 639).
  Lien trop parfait = artefact d'encodage probable. Ecartee par prudence.
- **`assigned_room_type`** : fuite de cible plus subtile. Quand la chambre attribuee differe de
  la reservee, le taux d'annulation tombe a 5,4 % contre 41,6 %. On n'attribue une chambre que si
  le client se presente : information connue apres la reservation. On garde seulement
  `reserved_room_type`, connue des le depart.

In [75]:
#df = df.drop(columns=["name", "email", "phone-number", "credit_card"])
df = df.drop(columns=["reservation_status", "reservation_status_date"])
df = df.drop(columns=["deposit_type"])
df = df.drop(columns=["assigned_room_type"])
print("Apres suppressions :", df.shape)

Apres suppressions : (119390, 28)


## 3. Valeurs manquantes

Chaque colonne est traitee selon sa logique metier :

- `company` (94 %) et `agent` (14 %) : un manque signifie "pas d'entreprise / pas d'agence".
  On garde l'info en booleen presence/absence.
- `country` (0,4 %) : categorielle, on cree une modalite "Unknown".
- `children` (4 lignes) : negligeable. **On supprime ces 4 lignes** plutot que d'imputer une
  valeur arbitraire.

In [76]:
df["has_company"] = df["company"].notna().astype(int)
df["has_agent"]   = df["agent"].notna().astype(int)
df = df.drop(columns=["company", "agent"])

df["country"] = df["country"].fillna("Unknown")

df = df.dropna(subset=["children"])        # suppression des 4 lignes
df["children"] = df["children"].astype(int)

print("Manquants restants :", df.isnull().sum().sum())

Manquants restants : 0


## 4. Valeurs aberrantes

On supprime uniquement les aberrations franches. Les cas ambigus (sejours a 0 nuit, adr a 0)
sont conserves car potentiellement legitimes.

- `adr < 0` : 1 ligne (prix impossible).
- `adr > 1000` : 1 ligne (5400, contre 510 pour la 2e valeur max).
- Reservations sans occupant : 0 adulte + 0 enfant + 0 bebe.

In [77]:
n0 = len(df)
df = df[df["adr"].between(0, 1000)]
no_guest = (df["adults"] == 0) & (df["children"] == 0) & (df["babies"] == 0)
df = df[~no_guest]
print("Lignes supprimees :", n0 - len(df))
print("Dimensions :", df.shape)

Lignes supprimees : 182
Dimensions : (119204, 28)


## 5. Correction des types

`arrival_date_month` est en texte non ordonne. On le convertit en categorie ordonnee Jan -> Dec,
pour en deriver la saison ensuite.

In [78]:
mois = ["January","February","March","April","May","June",
        "July","August","September","October","November","December"]
df["arrival_date_month"] = pd.Categorical(df["arrival_date_month"], categories=mois, ordered=True)

## 6. Creation de variables

Chaque feature repond a une observation de l'EDA :

- Binarisation des compteurs ecrases sur 0 (`had_previous_cancellation`, `has_special_requests`,
  `has_booking_changes`, `needs_parking`) : c'est surtout le "0 vs >=1" qui discrimine.
- `log_lead_time` : `lead_time` est tres asymetrique (le meilleur predicteur : 145j si annule
  contre 80j sinon), le log compresse la traine.
- `total_nights` : somme nuits semaine + week-end. Remplace `stays_in_weekend_nights`, non liee
  a l'annulation (Mann-Whitney p=0,16).
- `season` : le taux d'annulation varie selon la periode (effet modere, 33 % hiver a 39 % ete).

Pas de `room_changed` : elle reposait sur `assigned_room_type`, ecartee pour fuite de cible.

In [79]:
def add_features(d):
    d = d.copy()
    d["had_previous_cancellation"] = (d["previous_cancellations"] > 0).astype(int)
    d["has_special_requests"]      = (d["total_of_special_requests"] > 0).astype(int)
    d["has_booking_changes"]       = (d["booking_changes"] > 0).astype(int)
    d["needs_parking"]             = (d["required_car_parking_spaces"] > 0).astype(int)

    d["log_lead_time"] = np.log1p(d["lead_time"])
    d["total_nights"]  = d["stays_in_week_nights"] + d["stays_in_weekend_nights"]

    saison = {"December":"Hiver","January":"Hiver","February":"Hiver",
              "March":"Printemps","April":"Printemps","May":"Printemps",
              "June":"Ete","July":"Ete","August":"Ete",
              "September":"Automne","October":"Automne","November":"Automne"}
    d["season"] = d["arrival_date_month"].astype(str).map(saison)
    return d

df = add_features(df)
print("Colonnes apres feature engineering :", df.shape[1])

Colonnes apres feature engineering : 35


## 7. Regroupement des modalites rares

Les modalites ultra-rares creeraient des colonnes quasi vides (bruit). On regroupe :
- `market_segment` : Aviation, Complementary, Undefined -> "Other".
- `customer_type` : Group -> "Other".
- `meal` : Undefined -> "SC".

In [80]:
df["market_segment"] = df["market_segment"].replace(["Aviation","Complementary","Undefined"], "Other")
df["customer_type"]   = df["customer_type"].replace({"Group": "Other"})
df["meal"]            = df["meal"].replace({"Undefined": "SC"})

## 8. Suppression des colonnes redondantes

- `arrival_date_month` : remplacee par `season`.
- `stays_in_weekend_nights` : fusionnee dans `total_nights` (et non significative seule).

On conserve les compteurs bruts en plus de leurs versions binaires (utile pour les arbres), et
`children` malgre sa non-significativite univariee (peut jouer en interaction, cout nul).

In [81]:
df = df.drop(columns=["arrival_date_month", "stays_in_weekend_nights"])
print("Colonnes :", df.shape[1])

Colonnes : 33


## 9. Separation features / cible et split train/test

On splitte **avant** les encodages et le scaling (etapes 10-13), qui apprennent des donnees.
`stratify=y` conserve le ratio d'annulations (~37 %).

In [82]:
y = df["is_canceled"]
X = df.drop(columns=["is_canceled"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train :", X_train.shape, "| Test :", X_test.shape)

Train : (95363, 32) | Test : (23841, 32)


## 10. Encodage ordinal de reserved_room_type

Les types de chambre sont des codes A..L. Faute d'information metier sur leur hierarchie de gamme,
on applique un encodage ordinal (ordre alphabetique par defaut). **Choix provisoire** : si la
documentation confirme un ordre de gamme, on repassera en one-hot. L'encodeur est ajuste sur le
train seul ; une modalite inconnue du test recoit -1.

In [83]:
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[["reserved_room_type"]] = ord_enc.fit_transform(X_train[["reserved_room_type"]])
X_test[["reserved_room_type"]]  = ord_enc.transform(X_test[["reserved_room_type"]])
print("reserved_room_type encode (ordinal). Categories :", list(ord_enc.categories_[0]))

reserved_room_type encode (ordinal). Categories : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'L']


## 11. Encodage par frequence de country

`country` a 177 modalites : un one-hot serait ingerable. On remplace chaque pays par sa frequence
(calculee sur le train seul). C'est important car le pays est la 2e variable la plus liee a
l'annulation (chi2 = 15 619). Un pays absent du train recoit 0.

In [84]:
freq = X_train["country"].value_counts(normalize=True).to_dict()
X_train["country_freq"] = X_train["country"].map(freq)
X_test["country_freq"]  = X_test["country"].map(freq).fillna(0)
X_train = X_train.drop(columns=["country"])
X_test  = X_test.drop(columns=["country"])

## 12. Encodage one-hot des categorielles restantes

Categories nominales a faible cardinalite -> one-hot (`dtype=int` pour des 0/1, pas des
True/False). `drop_first=True` evite la colinearite. On aligne train et test pour des colonnes
identiques.

In [85]:
cat_cols = ["hotel", "meal", "market_segment", "distribution_channel",
            "customer_type", "season"]
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True, dtype=int)
X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True, dtype=int)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)
print("Colonnes apres one-hot :", X_train.shape[1])

Colonnes apres one-hot : 45


## 13. Sauvegarde du prix en euros (avant scaling)

Avant de standardiser, on met de cote `adr` en euros : il servira a l'analyse de couts metier.
On le capture maintenant car le scaling va le transformer.

In [86]:
adr_train_eur = X_train["adr"].copy()   # en euros, indexe par l'index d'origine
adr_test_eur  = X_test["adr"].copy()

## 14. Scaling des variables continues (optionnel selon le modele)

Standardisation des continues, fittee sur le train seul. A sauter pour les modeles a base
d'arbres (insensibles a l'echelle) ; utile pour la regression logistique, le SVM, le KNN.
Choix a tester en phase de modelisation.

In [87]:
num_to_scale = ["log_lead_time", "lead_time", "adr", "total_nights",
                "days_in_waiting_list", "arrival_date_week_number"]
num_to_scale = [c for c in num_to_scale if c in X_train.columns]

scaler = StandardScaler()
X_train[num_to_scale] = scaler.fit_transform(X_train[num_to_scale])
X_test[num_to_scale]  = scaler.transform(X_test[num_to_scale])
print("Variables standardisees :", num_to_scale)

Variables standardisees : ['log_lead_time', 'lead_time', 'adr', 'total_nights', 'days_in_waiting_list', 'arrival_date_week_number']


## 15. Fichier de reference metier (prix en euros)

On exporte a cote des fichiers modele une table de reference reliant chaque reservation (par son
index d'origine) a son prix en euros et a sa vraie cible. Elle servira UNIQUEMENT a l'analyse de
couts apres prediction (jamais comme feature : elle contient la cible).

In [91]:
ref = pd.concat([
    pd.DataFrame({"id": X_train.index, "adr_eur": adr_train_eur.values,
                  "is_canceled": y_train.values}),
    pd.DataFrame({"id": X_test.index,  "adr_eur": adr_test_eur.values,
                  "is_canceled": y_test.values}),
], ignore_index=True)

os.makedirs("data", exist_ok=True)
ref.to_csv("data/reference_prix.csv", index=False)
print("reference_prix.csv :", ref.shape)
print("adr moyen (EUR) :", round(ref["adr_eur"].mean(), 2))

reference_prix.csv : (119204, 3)
adr moyen (EUR) : 101.93


## 16. Split train / validation et export

On decoupe le train en train final + validation (le test reste intact pour la mesure finale).
On exporte avec `index=True` : l'index original est la cle de jointure avec `reference_prix.csv`.

In [90]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=42)

print("Train :", X_tr.shape, "| Val :", X_val.shape, "| Test :", X_test.shape)

X_tr.to_csv("data/X_train.csv", index=True)
X_val.to_csv("data/X_val.csv",  index=True)
X_test.to_csv("data/X_test.csv", index=True)
y_tr.to_csv("data/y_train.csv", index=True)
y_val.to_csv("data/y_val.csv",  index=True)
y_test.to_csv("data/y_test.csv", index=True)
print("6 fichiers modele + reference_prix.csv exportes dans data/")

Train : (71522, 45) | Val : (23841, 45) | Test : (23841, 45)
6 fichiers modele + reference_prix.csv exportes dans data/


## 17. Recapitulatif

Jeux prets pour la modelisation, toutes les decisions issues de l'EDA :

- **Supprime** : fuites de cible (reservation_status, assigned_room_type), fausses donnees perso,
  deposit_type (artefact), 4 lignes children manquantes, aberrations adr et sans occupant.
- **Manquants** : booleens (company/agent), "Unknown" (country).
- **Cree** : compteurs binarises, log_lead_time, total_nights, season.
- **Encode** : reserved_room_type en ordinal (provisoire), country par frequence, reste en one-hot 0/1.
- **Scaling** : continues, activable/desactivable selon le modele.

Rechargement cote modelisation : `pd.read_csv("data/X_train.csv", index_col=0)` et pour les
cibles `pd.read_csv("data/y_train.csv", index_col=0).squeeze("columns")`.